In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision.models import mobilenet_v2
import numpy as np
import matplotlib.pyplot as plt
import time
import random
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split
import pandas as pd
import copy


BASE_SEED = 42
LEARNING_RATE = 5e-4 
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1
EARLY_STOP_PATIENCE = 10
MAX_EPOCHS = 50 
BATCH_SIZE = 128


def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


class Swish(nn.Module):
    def forward(self, x):
        return x * torch.sigmoid(x)

class Mish(nn.Module):
    def forward(self, x):
        return x * torch.tanh(nn.functional.softplus(x))

class HSwish(nn.Module):
    def forward(self, x):
        return x * torch.clamp(x + 3, min=0, max=6) / 6

activations_dict = {
    'relu': nn.ReLU,
    'relu6': nn.ReLU6,
    'prelu': nn.PReLU,
    'mish': Mish,
    'swish': Swish,
    'elu': nn.ELU,
    'hswish': HSwish
}


def prepare_cifar10(batch_size=128):
    mean = (0.4914, 0.4822, 0.4465)
    std = (0.2023, 0.1994, 0.2010)
    
    transform_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ])
    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ])
    
    trainset_full = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
    testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)

    indices = list(range(len(trainset_full)))
    labels = [trainset_full[i][1] for i in indices]
    train_idx, val_idx = train_test_split(indices, test_size=0.2, 
                                           stratify=labels, random_state=BASE_SEED)
    
    trainset = torch.utils.data.Subset(trainset_full, train_idx)
    valset = torch.utils.data.Subset(trainset_full, val_idx)
    
    trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    valloader = torch.utils.data.DataLoader(valset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    
    return trainloader, valloader, testloader


def get_base_model_weights(num_classes=10, in_channels=3):
    set_seed(BASE_SEED)
    model = mobilenet_v2(weights=None)
    model.features[0][0] = nn.Conv2d(in_channels, 32, kernel_size=3, stride=1, padding=1, bias=False)
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)

    return model.state_dict()

def create_mobilenet_with_activation(activation_name, base_weights, num_classes=10, in_channels=3):
    model = mobilenet_v2(weights=None)
    model.features[0][0] = nn.Conv2d(in_channels, 32, kernel_size=3, stride=1, padding=1, bias=False)
    
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)

    model.load_state_dict(base_weights)

    activation_class = activations_dict[activation_name]

    for name, child in model.features.named_children():
        if isinstance(child, (nn.ReLU, nn.ReLU6, nn.PReLU)):
            setattr(model.features, name, activation_class())
        else:
            for sub_name, sub_child in child.named_children():
                if isinstance(sub_child, (nn.ReLU, nn.ReLU6, nn.PReLU)):
                    setattr(child, sub_name, activation_class())
  
    for name, child in model.classifier.named_children():
        if isinstance(child, (nn.ReLU, nn.ReLU6, nn.PReLU)):
            setattr(model.classifier, name, activation_class())
    
    return model


def train_model(model, trainloader, valloader, device, max_epochs=50, 
                early_stop_patience=10, lr=5e-4, weight_decay=1e-4):
    
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
    
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    
    best_val_loss = float('inf')
    best_epoch_by_loss = 0
    best_model_state = None
    patience_counter = 0
    
    for epoch in range(max_epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for inputs, labels in trainloader:
            inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
        
        train_loss = running_loss / len(trainloader.dataset)
        train_acc = 100. * correct / total
        
        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0
        
        with torch.no_grad():
            for inputs, labels in valloader:
                inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * inputs.size(0)
                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()
        
        val_loss = val_loss / len(valloader.dataset)
        val_acc = 100. * correct / total
        
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch_by_loss = epoch + 1
            best_model_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
        
        print(f"Epoch {epoch+1:2d}: Train Loss: {train_loss:.4f} Acc: {train_acc:.2f}% | "f"Val Loss: {val_loss:.4f} Acc: {val_acc:.2f}%")
        
        if patience_counter >= early_stop_patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    
    return model, history, best_epoch_by_loss


def measure_inference_time(model, testloader, device, num_measurements=30):
    model.eval()
    single_times = []
    
    with torch.no_grad():
        data_iter = iter(testloader)
        
        input_single, _ = next(data_iter)
        input_single = input_single[0:1].to(device, non_blocking=True)
        for _ in range(3):
            _ = model(input_single)
            torch.cuda.synchronize()

        for _ in range(num_measurements):
            try:
                inputs, _ = next(data_iter)
            except StopIteration:
                data_iter = iter(testloader)
                inputs, _ = next(data_iter)
            
            input_single = inputs[0:1].to(device, non_blocking=True)
            torch.cuda.synchronize()
            start = time.time()
            _ = model(input_single)
            torch.cuda.synchronize()
            single_times.append(time.time() - start)
    
    return np.mean(single_times)

def main():
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

    trainloader, valloader, testloader = prepare_cifar10(BATCH_SIZE)

    base_weights = get_base_model_weights(num_classes=10, in_channels=3)

    results = {act: {} for act in activations_dict.keys()}
    
    for act_name in activations_dict.keys():
        print(f"Активация: {act_name.upper()}")
        
        model = create_mobilenet_with_activation(act_name, base_weights, num_classes=10, in_channels=3).to(device)

        start_time = time.time()
        model, history, best_epoch = train_model(
            model, trainloader, valloader, device, 
            max_epochs=MAX_EPOCHS, early_stop_patience=EARLY_STOP_PATIENCE,
            lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
        )
        training_time = time.time() - start_time

        inf_time = measure_inference_time(model, testloader, device)

        all_preds = []
        all_labels = []
        with torch.no_grad():
            for inputs, labels in testloader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, preds = outputs.max(1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        
        acc = accuracy_score(all_labels, all_preds)
        f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)

        results[act_name] = {
            'best_epoch': best_epoch,
            'time': training_time,
            'acc': acc,
            'f1': f1,
            'inf_time': inf_time,
            'history': history
        }
        
        print(f"Best Epoch: {best_epoch}, Test Acc: {acc:.4f}, "f"F1: {f1:.4f}, Inf Time: {inf_time:.6f}s\n")

    
    rows = []
    for act_name in activations_dict.keys():
        rows.append({
            'Activation': act_name,
            'Best Val Epoch': int(results[act_name]['best_epoch']),
            'Training Time (s)': round(results[act_name]['time'], 2),
            'Test Acc': round(results[act_name]['acc'], 4),
            'Test F1': round(results[act_name]['f1'], 4),
            'Inference Time (s)': round(results[act_name]['inf_time'], 6)
        })
    
    df = pd.DataFrame(rows)
    print(df.to_string(index=False))
    
    df.to_csv('cifar10_results_final.csv', index=False)

    fig_train, axes_train = plt.subplots(1, 2, figsize=(14, 5))
    fig_train.suptitle('Сходимость (обучение) на CIFAR10', fontsize=16)
    
    for act_name in activations_dict.keys():
        history = results[act_name]['history']
        epochs = range(1, len(history['train_loss'])+1)
        axes_train[0].plot(epochs, history['train_loss'], label=act_name, linewidth=2)
        axes_train[1].plot(epochs, history['train_acc'], label=act_name, linewidth=2)
    
    axes_train[0].set_xlabel('Эпоха')
    axes_train[0].set_ylabel('Train Loss')
    axes_train[0].legend(fontsize=8, ncol=2)
    axes_train[0].grid(True, alpha=0.3)
    
    axes_train[1].set_xlabel('Эпоха')
    axes_train[1].set_ylabel('Train Accuracy (%)')
    axes_train[1].legend(fontsize=8, ncol=2)
    axes_train[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('train_convergence_cifar10.png', dpi=300, bbox_inches='tight')
    plt.show()

    fig_val, axes_val = plt.subplots(1, 2, figsize=(14, 5))
    fig_val.suptitle('Сходимость (валидация) на CIFAR10', fontsize=16)
    
    for act_name in activations_dict.keys():
        history = results[act_name]['history']
        epochs = range(1, len(history['val_loss'])+1)
        axes_val[0].plot(epochs, history['val_loss'], label=act_name, linewidth=2)
        axes_val[1].plot(epochs, history['val_acc'], label=act_name, linewidth=2)
    
    axes_val[0].set_xlabel('Эпоха')
    axes_val[0].set_ylabel('Validation Loss')
    axes_val[0].legend(fontsize=8, ncol=2)
    axes_val[0].grid(True, alpha=0.3)
    
    axes_val[1].set_xlabel('Эпоха')
    axes_val[1].set_ylabel('Validation Accuracy (%)')
    axes_val[1].legend(fontsize=8, ncol=2)
    axes_val[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('val_convergence_cifar10.png', dpi=300, bbox_inches='tight')
    plt.show()

if __name__ == "__main__":
    main()